In [ ]:

!pip uninstall -y -q tensorflow

!pip install -q --upgrade pip
!pip install -q transformers==4.40.0 accelerate==0.29.3 datasets==2.19.0 jiwer soundfile librosa

import os
import glob
import pandas as pd
import torch
from transformers import pipeline
from tqdm.notebook import tqdm

device = 0 if torch.cuda.is_available() else -1
print(f"Device set to: {'GPU' if device == 0 else 'CPU'}")


In [ ]:

MODEL_ID = "MODEL_ID"

pipe = pipeline(
    "automatic-speech-recognition", 
    model=MODEL_ID, 
    device=device 
)

print("Model loaded successfully.")

In [ ]:
pip install kaggle

In [ ]:

# Setup Kaggle credentials
import os
import json

!mkdir -p ~/.kaggle

!mv /workspace/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d anvitamanne/telugu-noisy-data --unzip -p /workspace/data

In [ ]:

print("--- Root Folders ---")
!ls -F /workspace/data

print("\n--- Finding Metadata CSV ---")
!find /workspace/data -name "*.csv"

In [ ]:
!apt-get update && apt-get install -y ffmpeg

In [ ]:
!ffmpeg -version

In [ ]:
import os
import glob
import pandas as pd
from transformers import pipeline
from tqdm.notebook import tqdm

DATASET_PATHS = {
    "noisy_dataset": "/workspace/data/archive (3)/noisy_dataset",
    "indictts_noise": "/workspace/data/archive (2)",
    "mozilla_noise": "/workspace/data/archive (1)"
}

results = []
limit = 5  

print(f"Starting inference (Test Limit: {limit})...")

for dataset_name, base_path in DATASET_PATHS.items():
    print(f"\nProcessing {dataset_name}...")
    
    files = []
    for ext in ['*.wav', '*.mp3', '*.flac']:
        files.extend(glob.glob(os.path.join(base_path, '**', ext), recursive=True))
    
    print(f"  Found {len(files)} files.")
    
    files_to_process = files[:limit] if limit else files
    
    for audio_path in tqdm(files_to_process, desc=f"Inference on {dataset_name}"):
        try:
            output = pipe(audio_path, generate_kwargs={"language": "telugu"})
            
            results.append({
                "dataset": dataset_name,
                "filename": os.path.basename(audio_path),
                "prediction": output["text"]
            })
        except Exception as e:
            print(f"Error on {os.path.basename(audio_path)}: {e}")

if results:
    preds_df = pd.DataFrame(results)
    preds_df.to_csv("whisper_predictions.csv", index=False)
    print(f"\nSaved {len(results)} predictions to 'whisper_predictions.csv'")
    
    pd.set_option('display.max_colwidth', None)
    print("\n--- PREDICTION PREVIEW ---")
    display(preds_df.head(10)) 
else:
    print("No predictions generated. Double check if the audio files are inside the archive folders.")

In [ ]:
!pip install -q jiwer

In [ ]:
import pandas as pd
import os
import glob


BASE_DIR = "/workspace/data"


MAPPING_CONFIG = {
    "mozilla": {
        "csv": os.path.join(BASE_DIR, "archive (1)/mozilla_transcripts_csv/transcripts.csv"),
        "audio_dir": os.path.join(BASE_DIR, "archive (1)/mozilla_noise")
    },
    "indictts": {
        "csv": os.path.join(BASE_DIR, "archive (2)/indictts_noise_transcripts.csv"),
        "audio_dir": os.path.join(BASE_DIR, "archive (2)/indictts_noise_audio")
    },
    "openslr": {
        "csv": os.path.join(BASE_DIR, "archive (3)/noisy_dataset/transcriptions.csv"),
        "audio_dir": os.path.join(BASE_DIR, "archive (3)/noisy_dataset/clips")
    }
}

master_data = []

for label, paths in MAPPING_CONFIG.items():
    if os.path.exists(paths['csv']):
        df = pd.read_csv(paths['csv'])
        df.columns = [c.lower().strip() for c in df.columns]

        
        file_col = next((c for c in df.columns if any(x in c for x in ['file', 'path', 'audio'])), None)
        text_col = next((c for c in df.columns if any(x in c for x in ['text', 'sentence', 'transcript'])), None)
        
        for _, row in df.iterrows():
            
            fname = os.path.basename(str(row[file_col]))
            
            full_audio_path = os.path.join(paths['audio_dir'], fname)
            
            if os.path.exists(full_audio_path):
                master_data.append({
                    "dataset": label,
                    "audio_path": full_audio_path,
                    "reference": str(row[text_col])
                })


final_df = pd.DataFrame(master_data)
print(f" Success! Found {len(final_df)} valid audio-text pairs across all archives.")
print(final_df.groupby('dataset').size())

In [ ]:
sample = final_df.sample(1).iloc[0]
print(f"Testing Path: {sample['audio_path']}")
print(f"Reference Text: {sample['reference']}")


out = pipe(sample['audio_path'], generate_kwargs={"language": "telugu"})
print(f"Whisper Prediction: {out['text']}")

In [ ]:
import pandas as pd
import os
import re
from jiwer import wer, cer
from tqdm.notebook import tqdm


def normalize_telugu(text):
    if not isinstance(text, str): return ""
        
    text = re.sub(r'[^\u0C00-\u0C7F\s]', '', text) 
    return re.sub(r'\s+', ' ', text).strip()

individual_results = []

print(f"Starting Full Evaluation of {len(final_df)} files...")

for idx, row in tqdm(final_df.iterrows(), total=len(final_df)):
    try:
        
        out = pipe(row['audio_path'], generate_kwargs={"language": "telugu"})

        
        ref_clean = normalize_telugu(row['reference'])
        hyp_clean = normalize_telugu(out["text"])
        
        individual_results.append({
            "dataset": row['dataset'],
            "reference": ref_clean,
            "prediction": hyp_clean,
            "wer": wer(ref_clean, hyp_clean) if ref_clean else 0,
            "cer": cer(ref_clean, hyp_clean) if ref_clean else 0
        })
    except Exception as e:
        continue


res_df = pd.DataFrame(individual_results)
pd.set_option('display.max_colwidth', None)

for ds in res_df['dataset'].unique():
    subset = res_df[res_df['dataset'] == ds]
    print(f"\n{'='*30} {ds.upper()} FULL REPORT {'='*30}")
    print(f"Total Samples: {len(subset)}")
    print(f"AVG WER: {subset['wer'].mean():.4f} ({subset['wer'].mean()*100:.2f}%)")
    print(f"AVG CER: {subset['cer'].mean():.4f} ({subset['cer'].mean()*100:.2f}%)")
    print("\nTOP 10 TRANSCRIPTS:")
    display(subset[['reference', 'prediction', 'wer']].head(10))

In [ ]:
from jiwer import wer, cer
import pandas as pd


all_refs = res_df['reference'].tolist()
all_hyps = res_df['prediction'].tolist()


total_corpus_wer = wer(all_refs, all_hyps)
total_corpus_cer = cer(all_refs, all_hyps)


summary_df = pd.DataFrame({
    "Total Files Processed": [len(res_df)],
    "Global Corpus WER": [f"{total_corpus_wer:.4f} ({total_corpus_wer*100:.2f}%)"],
    "Global Corpus CER": [f"{total_corpus_cer:.4f} ({total_corpus_cer*100:.2f}%)"]
})

print("\n" + "="*60)
print(" FINAL COMBINED EVALUATION ")
print("="*60)
display(summary_df)

# 3. 10 SAMPLE TRANSCRIPTS FROM COMBINED DATA
# Displaying 10 random samples from the total processed pool
print("\n10 SAMPLE TRANSCRIPTS")
pd.set_option('display.max_colwidth', None)
display(res_df.sample(10)[['dataset', 'reference', 'prediction']])

# 4. SAVE FINAL OUTPUT
# Save the full results locally to /workspace
res_df.to_csv("/workspace/whisper_full_corpus_results.csv", index=False)
print(f"\n Full evaluation saved to: /workspace/whisper_full_corpus_results.csv")